# Multi-Format Document Ingestion

Real knowledge bases rarely contain a single document type. A product team might have API docs in Markdown, meeting notes in DOCX, usage data in CSV, and a public website — all of which need to be searchable together. SynapseKit's loader family provides a consistent `aload()` interface across every format so you can mix and match without glue code.

**What you'll build:** A unified knowledge base that ingests five different source formats and answers questions across all of them.

**Time:** ~20 min | **Difficulty:** Intermediate

Guide: [Multi-Format Document Ingestion](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/multi-format-ingestion)

## Prerequisites & Setup

You will need:
- An **OpenAI API key** — set `OPENAI_API_KEY` in the env cell below
- Sample files for each format (PDF, DOCX, CSV) — update paths in the cells below
- `synapsekit`, `chromadb`, `pypdf`, and `python-docx` installed

**What you'll learn:**
- How to use `PDFLoader`, `DocxLoader`, `WebLoader`, `CSVLoader`, and `DirectoryLoader`
- How each loader attaches format-specific metadata automatically
- How to merge documents from different sources before splitting
- How `DirectoryLoader` recursively ingests an entire folder
- How to identify which source answered your question

In [ ]:
!pip install synapsekit chromadb pypdf python-docx -q

In [ ]:
import os

# Set your OpenAI API key before running the cells below
# os.environ['OPENAI_API_KEY'] = 'sk-...'

## Step 1: Load PDFs

`PDFLoader` returns one `Document` per page. Each document has `.page_content` and `.metadata` with `source` and `page` keys.

In [ ]:
from synapsekit.loaders import PDFLoader

# Page-level metadata lets you cite exact page numbers in answers.
loader = PDFLoader("docs/product-spec.pdf")
pdf_docs = await loader.aload()
print(f"PDF: {len(pdf_docs)} pages loaded")
# Each doc: metadata = {'source': 'docs/product-spec.pdf', 'page': N}

## Step 2: Load DOCX Files

`DocxLoader` extracts paragraphs in document order and attaches the filename. Headings are preserved as plain text so the splitter can use them as natural chunk boundaries.

In [ ]:
from synapsekit.loaders import DocxLoader

# DocxLoader extracts paragraphs in document order and attaches the filename.
# Headings are preserved as plain text so the splitter can use them as natural
# chunk boundaries.
loader = DocxLoader("docs/meeting-notes.docx")
docx_docs = await loader.aload()
print(f"DOCX: {len(docx_docs)} sections loaded")

## Step 3: Load Web Pages

`WebLoader` fetches the URL, strips HTML tags and navigation boilerplate, and returns clean prose. Useful for keeping docs in sync with a public site without copy-pasting content.

In [ ]:
from synapsekit.loaders import WebLoader

# WebLoader fetches the URL, strips HTML tags and navigation boilerplate,
# and returns clean prose.
loader = WebLoader("https://docs.example.com/api-reference")
web_docs = await loader.aload()
print(f"Web: {len(web_docs)} documents loaded")
# metadata = {'source': 'https://docs.example.com/api-reference'}

## Step 4: Load CSV Files

`CSVLoader` converts each row into a Document whose `page_content` is a key=value string representation. `content_columns` tells CSVLoader which columns to include in the text representation.

In [ ]:
from synapsekit.loaders import CSVLoader

# CSVLoader converts each row into a Document whose page_content is a
# key=value string representation. This lets the LLM reason over tabular
# data without needing a SQL layer.
loader = CSVLoader("data/usage-stats.csv", content_columns=["feature", "description"])
csv_docs = await loader.aload()
print(f"CSV: {len(csv_docs)} rows loaded")

## Step 5: Load an Entire Directory

`DirectoryLoader` walks the directory recursively and dispatches each file to the correct loader based on extension. One call replaces five manual loader instantiations when your docs are organized in a folder.

In [ ]:
from synapsekit.loaders import DirectoryLoader

# DirectoryLoader walks the directory recursively and dispatches each file
# to the correct loader based on extension.
loader = DirectoryLoader(
    "knowledge-base/",
    glob="**/*",                  # include all file types
    exclude=["**/*.pyc", "**/.DS_Store"],
)
dir_docs = await loader.aload()
print(f"Directory: {len(dir_docs)} documents loaded")

## Step 6: Merge All Sources

Concatenating all document lists before splitting means the splitter sees every document with its original metadata intact.

In [ ]:
# Concatenating all document lists before splitting means the splitter sees
# every document with its original metadata intact.
all_docs = pdf_docs + docx_docs + web_docs + csv_docs + dir_docs
print(f"Total documents before splitting: {len(all_docs)}")

## Step 7: Split and Ingest

Split all documents with a uniform chunking strategy, embed them, and store in Chroma for persistence.

In [ ]:
from synapsekit.splitters import RecursiveCharacterTextSplitter
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.vectorstores.chroma import ChromaVectorStore

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(all_docs)
print(f"Total chunks after splitting: {len(chunks)}")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = ChromaVectorStore(
    collection_name="unified-kb",
    embedding_function=embeddings,
    persist_directory="./chroma_db",
)
await vectorstore.aadd_documents(chunks)
print("Ingestion complete.")

## Step 8: Query Across All Formats

The `source` metadata key is set uniformly by every loader so you always know which document answered your question.

In [ ]:
from synapsekit import RAGPipeline
from synapsekit.llms.openai import OpenAILLM

rag = RAGPipeline(
    llm=OpenAILLM(model="gpt-4o-mini"),
    embeddings=embeddings,
    vectorstore=vectorstore,
)

answer, sources = await rag.aquery(
    "What features had the highest usage last quarter?",
    return_sources=True,
)

print("Answer:", answer)
print("\nSources used:")
for doc in sources:
    # 'source' is set by every loader so you always know the origin.
    print(f"  [{doc.metadata.get('source', 'unknown')}]")

## Complete Working Example

A single self-contained cell that loads all five formats, splits, ingests, and queries. Update the file paths to match your local files before running.

In [ ]:
import asyncio
from synapsekit import RAGPipeline
from synapsekit.loaders import PDFLoader, DocxLoader, WebLoader, CSVLoader, DirectoryLoader
from synapsekit.splitters import RecursiveCharacterTextSplitter
from synapsekit.embeddings.openai import OpenAIEmbeddings
from synapsekit.llms.openai import OpenAILLM
from synapsekit.vectorstores.chroma import ChromaVectorStore

async def load_all_sources():
    pdf_docs = await PDFLoader("docs/product-spec.pdf").aload()
    docx_docs = await DocxLoader("docs/meeting-notes.docx").aload()
    web_docs = await WebLoader("https://docs.example.com/api-reference").aload()
    csv_docs = await CSVLoader(
        "data/usage-stats.csv", content_columns=["feature", "description"]
    ).aload()
    dir_docs = await DirectoryLoader(
        "knowledge-base/", glob="**/*", exclude=["**/*.pyc"]
    ).aload()
    return pdf_docs + docx_docs + web_docs + csv_docs + dir_docs

async def main():
    all_docs = await load_all_sources()
    print(f"Loaded {len(all_docs)} documents total")

    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(all_docs)
    print(f"Split into {len(chunks)} chunks")

    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    vectorstore = ChromaVectorStore(
        collection_name="unified-kb",
        embedding_function=embeddings,
        persist_directory="./chroma_db",
    )
    await vectorstore.aadd_documents(chunks)

    rag = RAGPipeline(
        llm=OpenAILLM(model="gpt-4o-mini"),
        embeddings=embeddings,
        vectorstore=vectorstore,
    )

    answer, sources = await rag.aquery(
        "What features had the highest usage last quarter?",
        return_sources=True,
    )
    print("\nAnswer:", answer)
    print("\nSources:")
    for doc in sources:
        print(f"  - {doc.metadata.get('source', 'unknown')}")

asyncio.run(main())

## Next Steps

- [Choosing a Chunking Strategy](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/chunking-strategies) — tune splitting for different content types
- [Metadata Filtering in Vector Search](https://synapsekit.github.io/synapsekit-docs/docs/guides/rag/metadata-filtering) — query only from specific sources
- [Loaders reference](https://synapsekit.github.io/synapsekit-docs/docs/rag/loaders) — full list of available loaders and their options